---
tags: [tutorial]
---

# Classical Control Flow Patterns

Quantum circuits often have structure that depends on classical control flow: iterating over qubits, applying gates based on a graph's edges, or choosing between gate sequences. Qamomile supports these patterns through `qmc.range`, `qmc.items`, `if` branching, and `while` loops.

This chapter covers:

- `qmc.range()` for loops
- `qmc.items()` for iterating over dictionaries
- `if` and `while` on measurement results for mid-circuit branching

In [ ]:
# Install the latest Qamomile through pip!
# (Google Colab) Pick the line that matches your chosen Transpiler tab
# below and remove the leading "# " from it to run.
# !pip install qamomile                  # Qiskit (default)
# !pip install "qamomile[quri_parts]"    # QURI Parts
# !pip install "qamomile[cudaq-cu12]"    # CUDA-Q on a CUDA 12.x toolchain (use qamomile[cudaq-cu13] on CUDA 13.x). Linux / macOS-arm64 / WSL2 only.

This article uses Qiskit by default. Qamomile transpiles the same
`@qkernel` to multiple quantum SDKs, so you can follow it with another
SDK by swapping the import shown below — the rest of the article code
is identical regardless of the SDK you pick. On Colab, uncomment the
matching `pip install` line in the cell above first.

::::{tab-set}
:::{tab-item} Qiskit
:sync: qiskit

```python
from qamomile.qiskit import QiskitTranspiler

transpiler = QiskitTranspiler()
```
:::

:::{tab-item} QURI Parts
:sync: quri_parts

```python
from qamomile.quri_parts import QuriPartsTranspiler

transpiler = QuriPartsTranspiler()
```

**Heads-up — this article does not run end-to-end on QURI Parts.**
The later sections measure a qubit mid-circuit and then branch on
the result via `if bit:` / `while bit:`. The QURI Parts default
simulator (qulacs) does not expose mid-circuit measurement at the
public-API level, so the runtime control-flow demos will not
execute on this backend regardless of Qamomile's emit pass. Stick
with the Qiskit or CUDA-Q tab if you want to follow the article
end-to-end.
:::

:::{tab-item} CUDA-Q
:sync: cudaq

Use `qamomile[cudaq-cu12]` for a CUDA 12.x toolchain or
`qamomile[cudaq-cu13]` for a CUDA 13.x toolchain — pick the one that
matches your installed CUDA Toolkit. CUDA-Q is supported on Linux,
macOS arm64, and Windows-via-WSL2 only.

```python
from qamomile.cudaq import CudaqTranspiler

transpiler = CudaqTranspiler()
```
:::
::::

In [ ]:
# Transpiler — by default this article uses Qiskit. If you picked a
# different tab above (QURI Parts / CUDA-Q), copy the two lines from
# that tab into this cell in place of the two below, and make sure the
# matching pip install line further up has been uncommented.
from qamomile.qiskit import QiskitTranspiler

transpiler = QiskitTranspiler()

In [ ]:
import os

import qamomile.circuit as qmc

## `qmc.range` Loops

`qmc.range` may take `start`, `stop`, and `step` arguments. Here we create a qkernel that applies H to every other qubit and then entangles adjacent pairs with CX.

In [ ]:
@qmc.qkernel
def hadamard_chain(n: qmc.UInt) -> qmc.Vector[qmc.Bit]:
    q = qmc.qubit_array(n, name="q")

    # Apply H to every other qubit
    q[0::2] = qmc.h(q[0::2])

    # Entangle adjacent pairs
    for i in qmc.range(n - 1):
        q[i], q[i + 1] = qmc.cx(q[i], q[i + 1])

    return qmc.measure(q)

In [ ]:
hadamard_chain.draw(n=5, fold_loops=False)

:::{note}
The loop variable in `qmc.range` must be a **single variable** (e.g. `for i in qmc.range(n)`).
Tuple or list unpacking such as `for [i, j] in qmc.range(n)` is not supported and will raise a `SyntaxError`.
:::

## `qmc.items` for Sparse Interaction Data

Many quantum algorithms (QAOA, VQE) apply gates only on specific pairs of qubits, determined by a graph or interaction map. Rather than looping over all pairs, you can pass a **dictionary** of interactions and iterate with `qmc.items()`.

The dictionary type uses Qamomile's symbolic types: `qmc.Dict[qmc.Tuple[qmc.UInt, qmc.UInt], qmc.Float]` — keys are qubit index pairs, values are interaction weights.

In [ ]:
@qmc.qkernel
def sparse_coupling(
    n: qmc.UInt,
    edges: qmc.Dict[qmc.Tuple[qmc.UInt, qmc.UInt], qmc.Float],
    gamma: qmc.Float,
) -> qmc.Vector[qmc.Bit]:
    q = qmc.qubit_array(n, name="q")

    # Initial superposition
    for i in qmc.range(n):
        q[i] = qmc.h(q[i])

    # Apply RZZ interactions only on specified edges
    for (i, j), weight in qmc.items(edges):
        q[i], q[j] = qmc.rzz(q[i], q[j], gamma * weight)

    return qmc.measure(q)

:::{note}
`qmc.items` supports these loop patterns:

- `for key, value in qmc.items(d)` — scalar key
- `for (i, j), value in qmc.items(d)` — tuple key
- `for key, value in d.items()` — method-call form

The **value** target must be a single variable. Tuple unpacking in the value position
(e.g. `for _, (i, j) in qmc.items(d)`) is **not** supported and will raise a `SyntaxError`.
Similarly, single-target patterns like `for pair in qmc.items(d)` are not supported.
:::

## Inspecting with `transpiler.to_circuit()`

`draw()` does not yet support all patterns (particularly `items` with complex types, `if`, and `while`). In such cases, use `transpiler.to_circuit()` to see the concrete transpiled circuit after all parameters are bound.

In [ ]:
edge_data = {(0, 1): 1.0, (1, 2): -0.7, (0, 2): 0.3}

circuit = transpiler.to_circuit(
    sparse_coupling,
    bindings={"n": 3, "edges": edge_data, "gamma": 0.4},
)
# Qiskit's ``QuantumCircuit.__str__`` gives an ASCII diagram. Other
# SDKs return a different concrete type whose ``__str__`` is just an
# object ``repr`` — print the type name so this cell is SDK-portable.
print(type(circuit).__name__)

Only the three edges in `edge_data` produce RZZ gates — no wasted operations.

## `if` Branching and `while` Loops

Qamomile supports **mid-circuit measurement** followed by classical branching. The condition must be a **measurement result** (`Bit`), not an argument of qkernels.

This maps directly to hardware-level conditional execution: measure a qubit, then decide what to do next based on the outcome.

### `if` on a measurement result

A common pattern: measure one qubit and conditionally apply a gate to another qubit based on the outcome.

In [ ]:
@qmc.qkernel
def conditional_flip() -> qmc.Bit:
    q0 = qmc.qubit("q0")
    q1 = qmc.qubit("q1")

    q0 = qmc.x(q0)  # Prepare |1⟩
    bit = qmc.measure(q0)

    # Conditionally flip q1 based on q0's measurement
    if bit:
        q1 = qmc.x(q1)
    else:
        pass

    return qmc.measure(q1)

This transpiles to a Qiskit `if_else` instruction and can be executed:

In [ ]:
exe = transpiler.transpile(conditional_flip)
if os.environ.get("QAMOMILE_DOCS_TEST") == "1":
    print("Skipping dynamic-circuit execution in docs test mode.")
else:
    executor = transpiler.executor()
    job = exe.sample(executor, bindings={}, shots=100)
    result = job.result()
    for value, count in result.results:
        print(f"  bit={value}: {count} shots")

Since `q0` is prepared as |1⟩, the measurement always yields 1, so `q1` always gets flipped — every shot should return 1.

### `while` on a measurement result

A `while` loop repeats until the measurement condition becomes false. This is useful for repeat-until-success protocols.

In [ ]:
@qmc.qkernel
def repeat_until_zero() -> qmc.Bit:
    q = qmc.qubit("q")
    q = qmc.h(q)  # 50/50 chance of |0⟩ or |1⟩
    bit = qmc.measure(q)

    while bit:
        # Re-prepare and re-measure until we get 0
        q = qmc.qubit("q2")
        q = qmc.h(q)
        bit = qmc.measure(q)

    return bit

This transpiles to whatever runtime-loop primitive the backend
offers — Qiskit emits a `while_loop` instruction inside a
`QuantumCircuit`; CUDA-Q emits a `while:` block inside the
`@cudaq.kernel` source. We can inspect the generated circuit
structure (the printed type tells you which SDK-native object came
back):

In [ ]:
exe_while = transpiler.transpile(repeat_until_zero)
qc_while = exe_while.compiled_quantum[0].circuit
# As with the previous section, fall back to the type name so this
# cell is SDK-portable — Qiskit's QuantumCircuit prints an ASCII
# diagram, CUDA-Q's artifact prints a generic ``__repr__``.
print(type(qc_while).__name__)

### Combining `if` and `while`

You can combine both patterns. Here is a protocol that repeatedly measures and conditionally applies a correction gate:

In [ ]:
@qmc.qkernel
def measure_and_correct() -> qmc.Bit:
    q0 = qmc.qubit("q0")
    q1 = qmc.qubit("q1")

    q0 = qmc.h(q0)
    bit = qmc.measure(q0)

    while bit:
        # If bit is 1, apply correction to q1
        if bit:
            q1 = qmc.x(q1)
        else:
            q1 = q1
        # Re-prepare and re-measure
        q0 = qmc.qubit("q0_retry")
        q0 = qmc.h(q0)
        bit = qmc.measure(q0)

    return qmc.measure(q1)

In [ ]:
exe_combined = transpiler.transpile(measure_and_correct)
qc_combined = exe_combined.compiled_quantum[0].circuit
print(qc_combined)

## Summary

- `qmc.range(n)` for looping over symbolic ranges.
- `qmc.items(dict)` for iterating over sparse key-value data (edges, weights).
- `if bit:` and `while bit:` for branching on **measurement results**.
  Both branches must handle the same qubit handles (affine rule).
- These control flow patterns transpile to native quantum SDK instructions
  (e.g., Qiskit `if_else` and `while_loop`).

**Next**: [Reuse Patterns](08_reuse_patterns.ipynb) — helper qkernels, composite gates, and stub gates for top-down design.